In [1]:
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
import numpy as np
import pandas as pd
from app.pipeline.data_load import load_nfl_data
from app.pipeline.feature import add_match_result, add_last5_stat, add_prev_feature, add_last5_h2h_win_ratios ,add_historical_win_pct, add_pf_pa_by_season, add_home_away_team_avg_scores_before, add_league_avg_score_before, add_home_away_team_avg_stat_before, add_league_avg_stat_before
from app.pipeline.clean import remove_columns
from app.pipeline.glicko import add_glicko_features
from dotenv import load_dotenv


In [2]:
years = list(range(1999, 2026))
print(years)


[1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]


In [3]:
# data = load_nfl_data(seasons = years)

In [4]:
# data.write_csv("")

In [5]:
load_dotenv()
input_data_path = "/home/root495/Inexture/nfl-sport-prediction/backend/data/input/step2_nfl_cleaned_data.csv"

In [6]:
df = pd.read_csv(input_data_path)

In [7]:
df.isnull().sum()

game_id                            0
season                             0
game_date                          0
game_stadium                       0
home_team                          0
                                  ..
away_return_yards_negative         0
home_yards_after_catch_positive    0
away_yards_after_catch_positive    0
home_yards_after_catch_negative    0
away_yards_after_catch_negative    0
Length: 70, dtype: int64

In [8]:
df = remove_columns(df,["Unnamed: 0","total_away_raw_air_epa",
"home_field_goal_result_blocked",
"home_field_goal_result_made",
"home_field_goal_result_missed",
"away_field_goal_result_made",
"away_field_goal_result_blocked",
"away_field_goal_result_missed",
"home_extra_point_result_aborted",
"home_extra_point_result_blocked",
"home_extra_point_result_failed",
"home_extra_point_result_good",
"away_extra_point_result_aborted",
"away_extra_point_result_blocked",
"away_extra_point_result_failed",
"away_extra_point_result_good",
"home_third_down_converted.1",
"away_third_down_converted.1"])

KeyError: "['Unnamed: 0', 'total_away_raw_air_epa', 'home_field_goal_result_blocked', 'home_field_goal_result_made', 'home_field_goal_result_missed', 'away_field_goal_result_made', 'away_field_goal_result_blocked', 'away_field_goal_result_missed', 'home_extra_point_result_aborted', 'home_extra_point_result_blocked', 'home_extra_point_result_failed', 'home_extra_point_result_good', 'away_extra_point_result_aborted', 'away_extra_point_result_blocked', 'away_extra_point_result_failed', 'away_extra_point_result_good', 'home_third_down_converted.1', 'away_third_down_converted.1'] not found in axis"

In [9]:
df.columns.tolist()

['game_id',
 'season',
 'game_date',
 'game_stadium',
 'home_team',
 'away_team',
 'home_coach',
 'away_coach',
 'total_home_score',
 'total_away_score',
 'total_home_epa',
 'total_away_epa',
 'total_home_rush_epa',
 'total_away_rush_epa',
 'total_home_pass_epa',
 'total_away_pass_epa',
 'home_first_down_rush',
 'home_first_down_pass',
 'home_third_down_converted',
 'home_fourth_down_converted',
 'home_interception',
 'home_fumble_lost',
 'home_fumble_forced',
 'home_rush_attempt',
 'home_pass_attempt',
 'home_pass_touchdown',
 'home_qb_dropback',
 'home_rush_touchdown',
 'home_tackled_for_loss',
 'home_qb_hit',
 'home_punt_attempt',
 'home_kickoff_attempt',
 'home_kickoff_inside_twenty',
 'home_penalty_yards',
 'home_rushing_yards',
 'home_passing_yards',
 'home_receiving_yards',
 'home_yards_gained',
 'home_sack',
 'away_first_down_rush',
 'away_first_down_pass',
 'away_third_down_converted',
 'away_fourth_down_converted',
 'away_interception',
 'away_fumble_lost',
 'away_fumble_forc

In [10]:
df[df["total_home_score"] < df["total_away_score"]]

,game_id,season,game_date,game_stadium,home_team,away_team,home_coach,away_coach,total_home_score,total_away_score,...,away_yards_gained,away_sack,home_return_yards_positive,away_return_yards_positive,home_return_yards_negative,away_return_yards_negative,home_yards_after_catch_positive,away_yards_after_catch_positive,home_yards_after_catch_negative,away_yards_after_catch_negative
0,1999_01_ARI_PHI,1999,1999-09-12,Veterans Stadium,PHI,ARI,Andy Reid,Vince Tobin,24.0,25.0,...,1.0,1.0,184.0,196.0,0.0,0.0,0.0,0.0,0.0,0.0
5,1999_01_DAL_WAS,1999,1999-09-12,Jack Kent Cooke Stadium,WAS,DAL,Norv Turner,Chan Gailey,35.0,41.0,...,1.0,1.0,78.0,183.0,0.0,0.0,0.0,0.0,0.0,0.0
6,1999_01_DET_SEA,1999,1999-09-12,Seattle Kingdome,SEA,DET,Mike Holmgren,Bobby Ross,20.0,28.0,...,1.0,5.0,117.0,141.0,-1.0,-3.0,0.0,0.0,0.0,0.0
8,1999_01_MIA_DEN,1999,1999-09-13,Mile High Stadium,DEN,MIA,Mike Shanahan,Jimmy Johnson,21.0,38.0,...,1.0,0.0,105.0,58.0,0.0,0.0,9.0,0.0,0.0,0.0
9,1999_01_MIN_ATL,1999,1999-09-12,Georgia Dome,ATL,MIN,Dan Reeves,Dennis Green,14.0,17.0,...,1.0,0.0,67.0,94.0,-1.0,0.0,11.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7452,2025_18_WAS_PHI,2025,2026-01-04,Lincoln Financial Field,PHI,WAS,Nick Sirianni,Dan Quinn,17.0,24.0,...,1.0,0.0,84.0,42.0,0.0,0.0,95.0,86.0,0.0,0.0
7453,2025_19_BUF_JAX,2025,2026-01-11,TIAA Bank Stadium,JAX,BUF,Liam Coen,Sean McDermott,24.0,27.0,...,1.0,1.0,96.0,119.0,0.0,0.0,81.0,101.0,0.0,0.0
7455,2025_19_HOU_PIT,2025,2026-01-12,Acrisure Stadium,PIT,HOU,Mike Tomlin,DeMeco Ryans,6.0,30.0,...,1.0,3.0,172.0,73.0,0.0,0.0,75.0,90.0,0.0,0.0
7457,2025_19_LA_CAR,2025,2026-01-10,Bank of America Stadium,CAR,LA,Dave Canales,Sean McVay,31.0,34.0,...,1.0,1.0,118.0,73.0,-2.0,0.0,125.0,111.0,0.0,0.0


In [11]:
df  = add_match_result(df)

In [12]:
df = add_historical_win_pct(df)

In [13]:
df.columns.tolist()

['game_id',
 'season',
 'game_date',
 'game_stadium',
 'home_team',
 'away_team',
 'home_coach',
 'away_coach',
 'total_home_score',
 'total_away_score',
 'total_home_epa',
 'total_away_epa',
 'total_home_rush_epa',
 'total_away_rush_epa',
 'total_home_pass_epa',
 'total_away_pass_epa',
 'home_first_down_rush',
 'home_first_down_pass',
 'home_third_down_converted',
 'home_fourth_down_converted',
 'home_interception',
 'home_fumble_lost',
 'home_fumble_forced',
 'home_rush_attempt',
 'home_pass_attempt',
 'home_pass_touchdown',
 'home_qb_dropback',
 'home_rush_touchdown',
 'home_tackled_for_loss',
 'home_qb_hit',
 'home_punt_attempt',
 'home_kickoff_attempt',
 'home_kickoff_inside_twenty',
 'home_penalty_yards',
 'home_rushing_yards',
 'home_passing_yards',
 'home_receiving_yards',
 'home_yards_gained',
 'home_sack',
 'away_first_down_rush',
 'away_first_down_pass',
 'away_third_down_converted',
 'away_fourth_down_converted',
 'away_interception',
 'away_fumble_lost',
 'away_fumble_forc

In [14]:
df[(df["home_team"] == "BUF") | (df["away_team"] == "BUF")][["home_team", "away_team", "total_home_score", "total_away_score", "home_pct", "away_pct"]].head(20)

,home_team,away_team,total_home_score,total_away_score,home_pct,away_pct
12,IND,BUF,31.0,14.0,1.000000,0.000000
21,BUF,NYJ,17.0,3.0,0.500000,0.000000
34,BUF,PHI,26.0,0.0,0.666667,0.000000
58,MIA,BUF,18.0,23.0,0.666667,0.750000
63,BUF,PIT,24.0,21.0,0.800000,0.400000
77,BUF,LV,14.0,20.0,0.666667,0.500000
88,SEA,BUF,26.0,16.0,0.666667,0.571429
115,BAL,BUF,10.0,13.0,0.357143,0.625000
128,WAS,BUF,17.0,34.0,0.611111,0.666667
139,BUF,MIA,23.0,3.0,0.700000,0.777778


In [15]:
df = add_pf_pa_by_season(df)


In [16]:
df.columns.tolist()

['game_id',
 'season',
 'game_date',
 'game_stadium',
 'home_team',
 'away_team',
 'home_coach',
 'away_coach',
 'total_home_score',
 'total_away_score',
 'total_home_epa',
 'total_away_epa',
 'total_home_rush_epa',
 'total_away_rush_epa',
 'total_home_pass_epa',
 'total_away_pass_epa',
 'home_first_down_rush',
 'home_first_down_pass',
 'home_third_down_converted',
 'home_fourth_down_converted',
 'home_interception',
 'home_fumble_lost',
 'home_fumble_forced',
 'home_rush_attempt',
 'home_pass_attempt',
 'home_pass_touchdown',
 'home_qb_dropback',
 'home_rush_touchdown',
 'home_tackled_for_loss',
 'home_qb_hit',
 'home_punt_attempt',
 'home_kickoff_attempt',
 'home_kickoff_inside_twenty',
 'home_penalty_yards',
 'home_rushing_yards',
 'home_passing_yards',
 'home_receiving_yards',
 'home_yards_gained',
 'home_sack',
 'away_first_down_rush',
 'away_first_down_pass',
 'away_third_down_converted',
 'away_fourth_down_converted',
 'away_interception',
 'away_fumble_lost',
 'away_fumble_forc

In [17]:
df[["home_team","away_team","total_home_score","total_away_score","game_date"]]

,home_team,away_team,total_home_score,total_away_score,game_date
0,PHI,ARI,24.0,25.0,1999-09-12
1,JAX,SF,41.0,3.0,1999-09-12
2,CLE,PIT,0.0,43.0,1999-09-12
3,GB,LV,28.0,24.0,1999-09-12
4,TB,NYG,13.0,17.0,1999-09-12
...,...,...,...,...,...
7458,PIT,HOU,6.0,30.0,2026-01-12
7459,DEN,BUF,30.0,30.0,2026-01-17
7460,SEA,SF,41.0,6.0,2026-01-17
7461,CHI,LA,17.0,17.0,2026-01-18


In [18]:
df[((df["home_team"] == "PHI") | (df["away_team"] == "PHI")) & (df["season"] == 2018) ][["home_team", "away_team","total_home_score", "total_away_score","home_pf","home_pa", "away_pf", "away_pa", "game_date"]]  

,home_team,away_team,total_home_score,total_away_score,home_pf,home_pa,away_pf,away_pa,game_date
5200,PHI,ATL,18.0,12.0,18.0,12.0,12.0,18.0,2018-09-06
5217,TB,PHI,27.0,21.0,75.0,61.0,39.0,39.0,2018-09-16
5241,PHI,IND,20.0,16.0,59.0,55.0,60.0,63.0,2018-09-23
5253,TEN,PHI,20.0,20.0,69.0,70.0,79.0,75.0,2018-09-30
5268,PHI,MIN,21.0,23.0,100.0,98.0,142.0,160.0,2018-10-07
5278,NYG,PHI,13.0,34.0,117.0,162.0,134.0,111.0,2018-10-11
5304,PHI,CAR,17.0,21.0,151.0,132.0,142.0,131.0,2018-10-21
5312,JAX,PHI,18.0,24.0,134.0,170.0,175.0,150.0,2018-10-28
5345,PHI,DAL,20.0,27.0,195.0,177.0,181.0,168.0,2018-11-11
5356,NO,PHI,48.0,7.0,372.0,239.0,202.0,225.0,2018-11-18


In [19]:
df = add_home_away_team_avg_scores_before(df)
df = add_league_avg_score_before(df)

In [20]:
df.columns.tolist()

['game_id',
 'season',
 'game_date',
 'game_stadium',
 'home_team',
 'away_team',
 'home_coach',
 'away_coach',
 'total_home_score',
 'total_away_score',
 'total_home_epa',
 'total_away_epa',
 'total_home_rush_epa',
 'total_away_rush_epa',
 'total_home_pass_epa',
 'total_away_pass_epa',
 'home_first_down_rush',
 'home_first_down_pass',
 'home_third_down_converted',
 'home_fourth_down_converted',
 'home_interception',
 'home_fumble_lost',
 'home_fumble_forced',
 'home_rush_attempt',
 'home_pass_attempt',
 'home_pass_touchdown',
 'home_qb_dropback',
 'home_rush_touchdown',
 'home_tackled_for_loss',
 'home_qb_hit',
 'home_punt_attempt',
 'home_kickoff_attempt',
 'home_kickoff_inside_twenty',
 'home_penalty_yards',
 'home_rushing_yards',
 'home_passing_yards',
 'home_receiving_yards',
 'home_yards_gained',
 'home_sack',
 'away_first_down_rush',
 'away_first_down_pass',
 'away_third_down_converted',
 'away_fourth_down_converted',
 'away_interception',
 'away_fumble_lost',
 'away_fumble_forc

In [21]:
df[((df["home_team"] == "PHI") | (df["away_team"] == "PHI")) & (df["season"] == 2018) ][["home_team", "away_team","total_home_score", "total_away_score","home_team_avg_score","away_team_avg_score"]]  

,home_team,away_team,total_home_score,total_away_score,home_team_avg_score,away_team_avg_score
5200,PHI,ATL,18.0,12.0,18.000000,12.000000
5217,TB,PHI,27.0,21.0,37.500000,19.500000
5241,PHI,IND,20.0,16.0,19.666667,20.000000
5253,TEN,PHI,20.0,20.0,17.250000,19.750000
5268,PHI,MIN,21.0,23.0,20.000000,23.666667
5278,NYG,PHI,13.0,34.0,19.500000,22.333333
5304,PHI,CAR,17.0,21.0,21.571429,23.666667
5312,JAX,PHI,18.0,24.0,16.750000,21.875000
5345,PHI,DAL,20.0,27.0,21.666667,20.111111
5356,NO,PHI,48.0,7.0,37.200000,20.200000


In [22]:
eps = 1e-6

# Compute net rating features (log-transformed, with epsilon for numerical stability)
df["home_net_rating"] = np.log((df["home_pf"] + eps) / (df["home_pa"] + eps))
df["away_net_rating"] = np.log((df["away_pf"] + eps) / (df["away_pa"] + eps))
# Defensive efficiency: lower is better (log-transformed)
df["home_defense"] = np.log((df["home_pa"] + eps) / (df["home_pf"] + eps))
df["away_defense"] = np.log((df["away_pa"] + eps) / (df["away_pf"] + eps))

# Offensive efficiency for symmetry with defense ("offense" = PF/PA, log-transformed)
df["home_offense"] = np.log((df["home_pf"] + eps) / (df["home_pa"] + eps))
df["away_offense"] = np.log((df["away_pf"] + eps) / (df["away_pa"] + eps))



In [23]:
df.columns.tolist()

['game_id',
 'season',
 'game_date',
 'game_stadium',
 'home_team',
 'away_team',
 'home_coach',
 'away_coach',
 'total_home_score',
 'total_away_score',
 'total_home_epa',
 'total_away_epa',
 'total_home_rush_epa',
 'total_away_rush_epa',
 'total_home_pass_epa',
 'total_away_pass_epa',
 'home_first_down_rush',
 'home_first_down_pass',
 'home_third_down_converted',
 'home_fourth_down_converted',
 'home_interception',
 'home_fumble_lost',
 'home_fumble_forced',
 'home_rush_attempt',
 'home_pass_attempt',
 'home_pass_touchdown',
 'home_qb_dropback',
 'home_rush_touchdown',
 'home_tackled_for_loss',
 'home_qb_hit',
 'home_punt_attempt',
 'home_kickoff_attempt',
 'home_kickoff_inside_twenty',
 'home_penalty_yards',
 'home_rushing_yards',
 'home_passing_yards',
 'home_receiving_yards',
 'home_yards_gained',
 'home_sack',
 'away_first_down_rush',
 'away_first_down_pass',
 'away_third_down_converted',
 'away_fourth_down_converted',
 'away_interception',
 'away_fumble_lost',
 'away_fumble_forc

In [24]:
df["home_team_peformance"] = np.log(
    (df["home_team_avg_score"] + 1e-6) / (df["league_avg_score_before"] + 1e-6)
)
df["away_team_peformance"] = np.log(
    (df["away_team_avg_score"] + 1e-6) / (df["league_avg_score_before"] + 1e-6)
)


In [25]:
df = add_home_away_team_avg_stat_before(
    df,
    stat_name="passing_yards",
    home_stat_col="home_passing_yards",
    away_stat_col="away_passing_yards"
)


df = add_home_away_team_avg_stat_before(
    df,
    stat_name="receiving_yards",
    home_stat_col="home_receiving_yards",
    away_stat_col="away_receiving_yards"
)

df = add_home_away_team_avg_stat_before(
    df,
    stat_name="rushing_yards",
    home_stat_col="home_rushing_yards",
    away_stat_col="away_rushing_yards"
)

df = add_home_away_team_avg_stat_before(
    df,
    stat_name="yards_gained",
    home_stat_col="home_yards_gained",
    away_stat_col="away_yards_gained"
)



df = add_league_avg_stat_before(
    df,
    stat_name="passing_yards",
    home_stat_col="home_passing_yards",
    away_stat_col="away_passing_yards"
)


df = add_league_avg_stat_before(
    df,
    stat_name="receiving_yards",
    home_stat_col="home_receiving_yards",
    away_stat_col="away_receiving_yards"
)



df = add_league_avg_stat_before(
    df,
    stat_name="rushing_yards",
    home_stat_col="home_rushing_yards",
    away_stat_col="away_rushing_yards"
)

df = add_league_avg_stat_before(
    df, 
    stat_name="yards_gained",
    home_stat_col="home_yards_gained",
    away_stat_col="away_yards_gained"

)


In [26]:
df.columns.tolist()

['game_id',
 'season',
 'game_date',
 'game_stadium',
 'home_team',
 'away_team',
 'home_coach',
 'away_coach',
 'total_home_score',
 'total_away_score',
 'total_home_epa',
 'total_away_epa',
 'total_home_rush_epa',
 'total_away_rush_epa',
 'total_home_pass_epa',
 'total_away_pass_epa',
 'home_first_down_rush',
 'home_first_down_pass',
 'home_third_down_converted',
 'home_fourth_down_converted',
 'home_interception',
 'home_fumble_lost',
 'home_fumble_forced',
 'home_rush_attempt',
 'home_pass_attempt',
 'home_pass_touchdown',
 'home_qb_dropback',
 'home_rush_touchdown',
 'home_tackled_for_loss',
 'home_qb_hit',
 'home_punt_attempt',
 'home_kickoff_attempt',
 'home_kickoff_inside_twenty',
 'home_penalty_yards',
 'home_rushing_yards',
 'home_passing_yards',
 'home_receiving_yards',
 'home_yards_gained',
 'home_sack',
 'away_first_down_rush',
 'away_first_down_pass',
 'away_third_down_converted',
 'away_fourth_down_converted',
 'away_interception',
 'away_fumble_lost',
 'away_fumble_forc

In [27]:
df["home_avg_passing_yards"] = np.log((df["home_avg_passing_yards"] + 1e-6)/ (df["league_avg_passing_yards_before"] + 1e-6))
df["home_avg_receiving_yards"] = np.log((df["home_avg_receiving_yards"] + 1e-6 ) / (df["league_avg_receiving_yards_before"] + 1e-6))
df["home_avg_rushing_yards"] = np.log((df["home_avg_rushing_yards"] + 1e-6) / (df["league_avg_rushing_yards_before"] + 1e-6))
df["home_avg_yards_gained"] = np.log((df["home_avg_yards_gained"] + 1e-6) / (df["league_avg_yards_gained_before"] + 1e-6))


# df["away_avg_passing_yards_before"] = np.log((df["away_avg_passing_yards_before"] + 1e-6)/ (df["league_avg_passing_yards_before"] + 1e-6))
# df["away_avg_receiving_yards_before"] = np.log((df["away_avg_receiving_yards_before"] + 1e-6 ) / (df["league_avg_receiving_yards_before"] + 1e-6))
# df["away_avg_rushing_yards_before"] = np.log((df["away_avg_rushing_yards_before"] + 1e-6) / (df["league_avg_rushing_yards_before"] + 1e-6))
# df["away_avg_yards_gained_before"] = np.log((df["away_avg_yards_gained_before"] + 1e-6) / df["league_avg_yards_gained_before"])


In [28]:
df.columns

Index(['game_id', 'season', 'game_date', 'game_stadium', 'home_team',
       'away_team', 'home_coach', 'away_coach', 'total_home_score',
       'total_away_score',
       ...
       'home_avg_receiving_yards', 'away_avg_receiving_yards',
       'home_avg_rushing_yards', 'away_avg_rushing_yards',
       'home_avg_yards_gained', 'away_avg_yards_gained',
       'league_avg_passing_yards_before', 'league_avg_receiving_yards_before',
       'league_avg_rushing_yards_before', 'league_avg_yards_gained_before'],
      dtype='object', length=101)

In [29]:
# import pandas as pd
# import numpy as np

# from app.pipeline.inference import get_inputs


# def replace_05_with_model_prediction(
#     df: pd.DataFrame,
#     model,
#     scaler,
#     coach_encoder,
#     team_encoder,
#     ground_encoder,
#     result_col: str = "match_result"
# ) -> pd.DataFrame:
#     """
#     Replace rows where result == 0.5 using a trained model.

#     Workflow:
#     ---------
#     1. Find rows where result == 0.5
#     2. Extract match metadata
#     3. Generate input using get_inputs()
#     4. Encode categorical variables
#     5. Scale features
#     6. Predict using model
#     7. Replace 0.5 with 0 or 1

#     Returns:
#     --------
#     Updated DataFrame
#     """

#     df = df.copy()

#     # --------------------------------------------------
#     # Step 1: Find rows with result == 0.5
#     # --------------------------------------------------
#     mask = df[result_col] == 0.5
#     rows_to_update = df.loc[mask]

#     if rows_to_update.empty:
#         return df

#     # --------------------------------------------------
#     # Step 2: Iterate row by row
#     # --------------------------------------------------
#     for idx, row in rows_to_update.iterrows():

#         home_team = row["home_team"]
#         away_team = row["away_team"]
#         home_coach = row["home_coach"]
#         away_coach = row["away_coach"]
#         game_stadium = row["game_stadium"]

#         # --------------------------------------------------
#         # Step 3: Build model input
#         # --------------------------------------------------
#         input_df = get_inputs(
#             home_team=home_team,
#             away_team=away_team,
#             home_coach=home_coach,
#             away_coach=away_coach,
#             game_stadium=game_stadium
#         )

#         # --------------------------------------------------
#         # Safety check (important for production)
#         # --------------------------------------------------
#         required_cols = {
#             "home_team",
#             "away_team",
#             "home_coach",
#             "away_coach",
#             "game_stadium"
#         }

#         missing = required_cols - set(input_df.columns)
#         if missing:
#             raise KeyError(f"Missing required columns in model input: {missing}")

#         # --------------------------------------------------
#         # Step 4: Encode categorical columns
#         # --------------------------------------------------
#         input_df["home_team"] = team_encoder.transform(input_df["home_team"])
#         input_df["away_team"] = team_encoder.transform(input_df["away_team"])

#         input_df["home_coach"] = coach_encoder.transform(input_df["home_coach"])
#         input_df["away_coach"] = coach_encoder.transform(input_df["away_coach"])

#         input_df["game_stadium"] = ground_encoder.transform(
#             input_df["game_stadium"]
#         )

#         # --------------------------------------------------
#         # Step 5: Scale features
#         # --------------------------------------------------
#         X_scaled = scaler.transform(input_df)

#         # --------------------------------------------------
#         # Step 6: Predict
#         # --------------------------------------------------
#         if hasattr(model, "predict_proba"):
#             pred = int(model.predict_proba(X_scaled)[0, 1] >= 0.5)
#         else:
#             pred = int(model.predict(X_scaled)[0])

#         # --------------------------------------------------
#         # Step 7: Replace 0.5 with prediction
#         # --------------------------------------------------
#         df.at[idx, result_col] = pred

#     return df


In [30]:
# import pickle
# with open("/home/root495/Inexture/nfl-sport-prediction/backend/models/coach_encoder.pkl", "rb") as f:
#     coach_le = pickle.load(f)

# with open("/home/root495/Inexture/nfl-sport-prediction/backend/models/team_encoder.pkl", "rb") as f:
#     team_le = pickle.load(f)

# with open("/home/root495/Inexture/nfl-sport-prediction/backend/models/ground_encoder.pkl", "rb") as f:
#     ground_le = pickle.load(f)

# with open("/home/root495/Inexture/nfl-sport-prediction/backend/models/scaler.pkl", "rb") as f:
#     scaler = pickle.load(f)
# model = None
# with open("/home/root495/Inexture/nfl-sport-prediction/backend/models/cat_model.pkl", "rb") as f:
#     model = pickle.load(f)

In [31]:
# df_updated = replace_05_with_model_prediction(
#     df=df,
#     model=model,
#     scaler=scaler,
#     coach_encoder=coach_le,
#     team_encoder=team_le,
#     ground_encoder=ground_le
# )


In [32]:
# df_updated["match_result"].value_counts()

In [33]:
# # df = df[~(df["match_result"] == 0.5)]
# df = df_updated

In [34]:
home_list = []
away_list = []
for i in df.columns.tolist():
    if "home" in i:
        home_list.append(i)
    if "away" in i:
        away_list.append(i)
temp_df = pd.DataFrame({
    "home" : home_list[2:],
    "away" : away_list[2:]
})

In [35]:
home_list[2:]

['total_home_score',
 'total_home_epa',
 'total_home_rush_epa',
 'total_home_pass_epa',
 'home_first_down_rush',
 'home_first_down_pass',
 'home_third_down_converted',
 'home_fourth_down_converted',
 'home_interception',
 'home_fumble_lost',
 'home_fumble_forced',
 'home_rush_attempt',
 'home_pass_attempt',
 'home_pass_touchdown',
 'home_qb_dropback',
 'home_rush_touchdown',
 'home_tackled_for_loss',
 'home_qb_hit',
 'home_punt_attempt',
 'home_kickoff_attempt',
 'home_kickoff_inside_twenty',
 'home_penalty_yards',
 'home_rushing_yards',
 'home_passing_yards',
 'home_receiving_yards',
 'home_yards_gained',
 'home_sack',
 'home_return_yards_positive',
 'home_return_yards_negative',
 'home_yards_after_catch_positive',
 'home_yards_after_catch_negative',
 'home_pct',
 'home_pf',
 'home_pa',
 'home_team_avg_score',
 'home_net_rating',
 'home_defense',
 'home_offense',
 'home_team_peformance',
 'home_avg_passing_yards',
 'home_avg_receiving_yards',
 'home_avg_rushing_yards',
 'home_avg_yard

In [36]:
away_list[2:]

['total_away_score',
 'total_away_epa',
 'total_away_rush_epa',
 'total_away_pass_epa',
 'away_first_down_rush',
 'away_first_down_pass',
 'away_third_down_converted',
 'away_fourth_down_converted',
 'away_interception',
 'away_fumble_lost',
 'away_fumble_forced',
 'away_rush_attempt',
 'away_pass_attempt',
 'away_pass_touchdown',
 'away_qb_dropback',
 'away_rush_touchdown',
 'away_tackled_for_loss',
 'away_qb_hit',
 'away_punt_attempt',
 'away_kickoff_attempt',
 'away_kickoff_inside_twenty',
 'away_penalty_yards',
 'away_rushing_yards',
 'away_passing_yards',
 'away_receiving_yards',
 'away_yards_gained',
 'away_sack',
 'away_return_yards_positive',
 'away_return_yards_negative',
 'away_yards_after_catch_positive',
 'away_yards_after_catch_negative',
 'away_pct',
 'away_pf',
 'away_pa',
 'away_team_avg_score',
 'away_net_rating',
 'away_defense',
 'away_offense',
 'away_team_peformance',
 'away_avg_passing_yards',
 'away_avg_receiving_yards',
 'away_avg_rushing_yards',
 'away_avg_yard

In [37]:
for i in temp_df.itertuples():
    df = add_last5_stat(df,i.home, i.away)

In [38]:
for i in df.columns:
    print(i)

game_id
season
game_date
game_stadium
home_team
away_team
home_coach
away_coach
total_home_score
total_away_score
total_home_epa
total_away_epa
total_home_rush_epa
total_away_rush_epa
total_home_pass_epa
total_away_pass_epa
home_first_down_rush
home_first_down_pass
home_third_down_converted
home_fourth_down_converted
home_interception
home_fumble_lost
home_fumble_forced
home_rush_attempt
home_pass_attempt
home_pass_touchdown
home_qb_dropback
home_rush_touchdown
home_tackled_for_loss
home_qb_hit
home_punt_attempt
home_kickoff_attempt
home_kickoff_inside_twenty
home_penalty_yards
home_rushing_yards
home_passing_yards
home_receiving_yards
home_yards_gained
home_sack
away_first_down_rush
away_first_down_pass
away_third_down_converted
away_fourth_down_converted
away_interception
away_fumble_lost
away_fumble_forced
away_rush_attempt
away_pass_attempt
away_pass_touchdown
away_qb_dropback
away_rush_touchdown
away_tackled_for_loss
away_qb_hit
away_punt_attempt
away_kickoff_attempt
away_kickoff_

In [39]:
df

,game_id,season,game_date,game_stadium,home_team,away_team,home_coach,away_coach,total_home_score,total_away_score,...,last_5_home_team_peformance,last_5_away_team_peformance,last_5_home_avg_passing_yards,last_5_away_avg_passing_yards,last_5_home_avg_receiving_yards,last_5_away_avg_receiving_yards,last_5_home_avg_rushing_yards,last_5_away_avg_rushing_yards,last_5_home_avg_yards_gained,last_5_away_avg_yards_gained
0,1999_01_ARI_PHI,1999,1999-09-12,Veterans Stadium,PHI,ARI,Andy Reid,Vince Tobin,24.0,25.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0
1,1999_01_SF_JAX,1999,1999-09-12,Alltel Stadium,JAX,SF,Tom Coughlin,Steve Mariucci,41.0,3.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0
2,1999_01_PIT_CLE,1999,1999-09-12,Cleveland Browns Stadium,CLE,PIT,Chris Palmer,Bill Cowher,0.0,43.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0
3,1999_01_OAK_GB,1999,1999-09-12,Lambeau Field,GB,LV,Ray Rhodes,Jon Gruden,28.0,24.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0
4,1999_01_NYG_TB,1999,1999-09-12,Raymond James Stadium,TB,NYG,Tony Dungy,Jim Fassel,13.0,17.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7458,2025_19_HOU_PIT,2025,2026-01-12,Acrisure Stadium,PIT,HOU,Mike Tomlin,DeMeco Ryans,6.0,30.0,...,0.024395,-0.010267,0.445410,0.319608,0.445958,0.320404,0.148701,0.077471,0.6,0.4
7459,2025_20_BUF_DEN,2025,2026-01-17,Empower Field at Mile High,DEN,BUF,Sean Payton,Sean McDermott,30.0,30.0,...,0.027153,0.210176,0.244260,0.329805,0.245036,0.316345,0.124618,0.326780,0.4,0.6
7460,2025_20_SF_SEA,2025,2026-01-17,Lumen Field,SEA,SF,Mike Macdonald,Kyle Shanahan,41.0,6.0,...,0.222223,0.101723,0.408875,0.320360,0.409368,0.317255,0.182706,0.012890,0.6,0.4
7461,2025_20_LA_CHI,2025,2026-01-18,Soldier Field,CHI,LA,Ben Johnson,Sean McVay,17.0,17.0,...,0.110904,0.259280,0.061732,0.421939,0.057599,0.422473,0.139612,0.155998,0.2,0.6


In [40]:
df = add_glicko_features(df)

In [41]:
df.columns.tolist()

['game_id',
 'season',
 'game_date',
 'game_stadium',
 'home_team',
 'away_team',
 'home_coach',
 'away_coach',
 'total_home_score',
 'total_away_score',
 'total_home_epa',
 'total_away_epa',
 'total_home_rush_epa',
 'total_away_rush_epa',
 'total_home_pass_epa',
 'total_away_pass_epa',
 'home_first_down_rush',
 'home_first_down_pass',
 'home_third_down_converted',
 'home_fourth_down_converted',
 'home_interception',
 'home_fumble_lost',
 'home_fumble_forced',
 'home_rush_attempt',
 'home_pass_attempt',
 'home_pass_touchdown',
 'home_qb_dropback',
 'home_rush_touchdown',
 'home_tackled_for_loss',
 'home_qb_hit',
 'home_punt_attempt',
 'home_kickoff_attempt',
 'home_kickoff_inside_twenty',
 'home_penalty_yards',
 'home_rushing_yards',
 'home_passing_yards',
 'home_receiving_yards',
 'home_yards_gained',
 'home_sack',
 'away_first_down_rush',
 'away_first_down_pass',
 'away_third_down_converted',
 'away_fourth_down_converted',
 'away_interception',
 'away_fumble_lost',
 'away_fumble_forc

In [42]:
df

,game_id,season,game_date,game_stadium,home_team,away_team,home_coach,away_coach,total_home_score,total_away_score,...,last_5_home_avg_rushing_yards,last_5_away_avg_rushing_yards,last_5_home_avg_yards_gained,last_5_away_avg_yards_gained,home_team_glicko_rating,home_team_rd,home_team_vol,away_team_glicko_rating,away_team_rd,away_team_vol
0,1999_01_ARI_PHI,1999,1999-09-12,Veterans Stadium,PHI,ARI,Andy Reid,Vince Tobin,24.0,25.0,...,0.000000,0.000000,0.0,0.0,1500.000000,350.000000,0.060000,1500.000000,350.000000,0.060000
1,1999_01_SF_JAX,1999,1999-09-12,Alltel Stadium,JAX,SF,Tom Coughlin,Steve Mariucci,41.0,3.0,...,0.000000,0.000000,0.0,0.0,1500.000000,350.000000,0.060000,1500.000000,350.000000,0.060000
2,1999_01_PIT_CLE,1999,1999-09-12,Cleveland Browns Stadium,CLE,PIT,Chris Palmer,Bill Cowher,0.0,43.0,...,0.000000,0.000000,0.0,0.0,1500.000000,350.000000,0.060000,1500.000000,350.000000,0.060000
3,1999_01_OAK_GB,1999,1999-09-12,Lambeau Field,GB,LV,Ray Rhodes,Jon Gruden,28.0,24.0,...,0.000000,0.000000,0.0,0.0,1500.000000,350.000000,0.060000,1500.000000,350.000000,0.060000
4,1999_01_NYG_TB,1999,1999-09-12,Raymond James Stadium,TB,NYG,Tony Dungy,Jim Fassel,13.0,17.0,...,0.000000,0.000000,0.0,0.0,1500.000000,350.000000,0.060000,1500.000000,350.000000,0.060000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7458,2025_19_HOU_PIT,2025,2026-01-12,Acrisure Stadium,PIT,HOU,Mike Tomlin,DeMeco Ryans,6.0,30.0,...,0.148701,0.077471,0.6,0.4,1575.089006,61.291731,0.059956,1580.875710,61.845895,0.059928
7459,2025_20_BUF_DEN,2025,2026-01-17,Empower Field at Mile High,DEN,BUF,Sean Payton,Sean McDermott,30.0,30.0,...,0.124618,0.326780,0.4,0.6,1598.463175,61.768175,0.059948,1700.208478,65.045688,0.059933
7460,2025_20_SF_SEA,2025,2026-01-17,Lumen Field,SEA,SF,Mike Macdonald,Kyle Shanahan,41.0,6.0,...,0.182706,0.012890,0.6,0.4,1625.701141,61.699388,0.059867,1596.396031,61.927486,0.059896
7461,2025_20_LA_CHI,2025,2026-01-18,Soldier Field,CHI,LA,Ben Johnson,Sean McVay,17.0,17.0,...,0.139612,0.155998,0.2,0.6,1504.148012,62.093637,0.059885,1635.624851,62.458269,0.059940


In [43]:
home_list = []
away_list = []
for i in df.columns:
    if "last_5" not in i:
        if "home" in i:
            home_list.append(i)

        if "away" in i:
            away_list.append(i)

temp_df = pd.DataFrame({
    "home" : home_list[2:],
    "away" : away_list[2:]
})

In [44]:
home_list[2:]

['total_home_score',
 'total_home_epa',
 'total_home_rush_epa',
 'total_home_pass_epa',
 'home_first_down_rush',
 'home_first_down_pass',
 'home_third_down_converted',
 'home_fourth_down_converted',
 'home_interception',
 'home_fumble_lost',
 'home_fumble_forced',
 'home_rush_attempt',
 'home_pass_attempt',
 'home_pass_touchdown',
 'home_qb_dropback',
 'home_rush_touchdown',
 'home_tackled_for_loss',
 'home_qb_hit',
 'home_punt_attempt',
 'home_kickoff_attempt',
 'home_kickoff_inside_twenty',
 'home_penalty_yards',
 'home_rushing_yards',
 'home_passing_yards',
 'home_receiving_yards',
 'home_yards_gained',
 'home_sack',
 'home_return_yards_positive',
 'home_return_yards_negative',
 'home_yards_after_catch_positive',
 'home_yards_after_catch_negative',
 'home_pct',
 'home_pf',
 'home_pa',
 'home_team_avg_score',
 'home_net_rating',
 'home_defense',
 'home_offense',
 'home_team_peformance',
 'home_avg_passing_yards',
 'home_avg_receiving_yards',
 'home_avg_rushing_yards',
 'home_avg_yard

In [45]:
away_list[2:]

['total_away_score',
 'total_away_epa',
 'total_away_rush_epa',
 'total_away_pass_epa',
 'away_first_down_rush',
 'away_first_down_pass',
 'away_third_down_converted',
 'away_fourth_down_converted',
 'away_interception',
 'away_fumble_lost',
 'away_fumble_forced',
 'away_rush_attempt',
 'away_pass_attempt',
 'away_pass_touchdown',
 'away_qb_dropback',
 'away_rush_touchdown',
 'away_tackled_for_loss',
 'away_qb_hit',
 'away_punt_attempt',
 'away_kickoff_attempt',
 'away_kickoff_inside_twenty',
 'away_penalty_yards',
 'away_rushing_yards',
 'away_passing_yards',
 'away_receiving_yards',
 'away_yards_gained',
 'away_sack',
 'away_return_yards_positive',
 'away_return_yards_negative',
 'away_yards_after_catch_positive',
 'away_yards_after_catch_negative',
 'away_pct',
 'away_pf',
 'away_pa',
 'away_team_avg_score',
 'away_net_rating',
 'away_defense',
 'away_offense',
 'away_team_peformance',
 'away_avg_passing_yards',
 'away_avg_receiving_yards',
 'away_avg_rushing_yards',
 'away_avg_yard

In [46]:
for i in temp_df.itertuples():
    df = add_prev_feature(df, i.home, i.away)

In [47]:
for i in df.columns:
    print(i)

game_id
season
game_date
game_stadium
home_team
away_team
home_coach
away_coach
total_home_score
total_away_score
total_home_epa
total_away_epa
total_home_rush_epa
total_away_rush_epa
total_home_pass_epa
total_away_pass_epa
home_first_down_rush
home_first_down_pass
home_third_down_converted
home_fourth_down_converted
home_interception
home_fumble_lost
home_fumble_forced
home_rush_attempt
home_pass_attempt
home_pass_touchdown
home_qb_dropback
home_rush_touchdown
home_tackled_for_loss
home_qb_hit
home_punt_attempt
home_kickoff_attempt
home_kickoff_inside_twenty
home_penalty_yards
home_rushing_yards
home_passing_yards
home_receiving_yards
home_yards_gained
home_sack
away_first_down_rush
away_first_down_pass
away_third_down_converted
away_fourth_down_converted
away_interception
away_fumble_lost
away_fumble_forced
away_rush_attempt
away_pass_attempt
away_pass_touchdown
away_qb_dropback
away_rush_touchdown
away_tackled_for_loss
away_qb_hit
away_punt_attempt
away_kickoff_attempt
away_kickoff_

In [48]:
df = add_last5_h2h_win_ratios(df)


In [49]:
df.columns.tolist()

['game_id',
 'season',
 'game_date',
 'game_stadium',
 'home_team',
 'away_team',
 'home_coach',
 'away_coach',
 'total_home_score',
 'total_away_score',
 'total_home_epa',
 'total_away_epa',
 'total_home_rush_epa',
 'total_away_rush_epa',
 'total_home_pass_epa',
 'total_away_pass_epa',
 'home_first_down_rush',
 'home_first_down_pass',
 'home_third_down_converted',
 'home_fourth_down_converted',
 'home_interception',
 'home_fumble_lost',
 'home_fumble_forced',
 'home_rush_attempt',
 'home_pass_attempt',
 'home_pass_touchdown',
 'home_qb_dropback',
 'home_rush_touchdown',
 'home_tackled_for_loss',
 'home_qb_hit',
 'home_punt_attempt',
 'home_kickoff_attempt',
 'home_kickoff_inside_twenty',
 'home_penalty_yards',
 'home_rushing_yards',
 'home_passing_yards',
 'home_receiving_yards',
 'home_yards_gained',
 'home_sack',
 'away_first_down_rush',
 'away_first_down_pass',
 'away_third_down_converted',
 'away_fourth_down_converted',
 'away_interception',
 'away_fumble_lost',
 'away_fumble_forc

In [50]:
# df = df[~(df["match_result"] == 0.5)]


In [51]:
df["match_result"].value_counts(
    
)

1.0    3919
0.0    3062
0.5     482
Name: match_result, dtype: int64

In [52]:
df.to_csv("/home/root495/Inexture/nfl-sport-prediction/backend/data/input/step3_nfl_processed_data.csv") 

In [53]:
df

,game_id,season,game_date,game_stadium,home_team,away_team,home_coach,away_coach,total_home_score,total_away_score,...,prev_home_avg_yards_gained,prev_away_avg_yards_gained,prev_home_team_glicko_rating,prev_away_team_glicko_rating,prev_home_team_rd,prev_away_team_rd,prev_home_team_vol,prev_away_team_vol,h2h_home_win_ratio,h2h_away_win_ratio
0,1999_01_ARI_PHI,1999,1999-09-12,Veterans Stadium,PHI,ARI,Andy Reid,Vince Tobin,24.0,25.0,...,0.0,1.0,1513.558175,1514.756649,68.080799,68.047382,0.059965,0.059965,0.0,0.0
1,1999_01_SF_JAX,1999,1999-09-12,Alltel Stadium,JAX,SF,Tom Coughlin,Steve Mariucci,41.0,3.0,...,0.0,1.0,1513.558175,1514.756649,68.080799,68.047382,0.059965,0.059965,0.0,0.0
2,1999_01_PIT_CLE,1999,1999-09-12,Cleveland Browns Stadium,CLE,PIT,Chris Palmer,Bill Cowher,0.0,43.0,...,0.0,1.0,1513.558175,1514.756649,68.080799,68.047382,0.059965,0.059965,0.0,0.0
3,1999_01_OAK_GB,1999,1999-09-12,Lambeau Field,GB,LV,Ray Rhodes,Jon Gruden,28.0,24.0,...,0.0,1.0,1513.558175,1514.756649,68.080799,68.047382,0.059965,0.059965,0.0,0.0
4,1999_01_NYG_TB,1999,1999-09-12,Raymond James Stadium,TB,NYG,Tony Dungy,Jim Fassel,13.0,17.0,...,0.0,1.0,1513.558175,1514.756649,68.080799,68.047382,0.059965,0.059965,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7458,2025_19_HOU_PIT,2025,2026-01-12,Acrisure Stadium,PIT,HOU,Mike Tomlin,DeMeco Ryans,6.0,30.0,...,0.0,0.0,1563.213624,1573.095618,61.339504,61.865037,0.059955,0.059929,0.6,0.4
7459,2025_20_BUF_DEN,2025,2026-01-17,Empower Field at Mile High,DEN,BUF,Sean Payton,Sean McDermott,30.0,30.0,...,0.0,1.0,1588.421870,1694.250626,61.855407,65.055276,0.059949,0.059935,0.2,0.8
7460,2025_20_SF_SEA,2025,2026-01-17,Lumen Field,SEA,SF,Mike Macdonald,Kyle Shanahan,41.0,6.0,...,1.0,1.0,1615.940716,1583.226667,61.780341,61.982002,0.059868,0.059895,0.4,0.6
7461,2025_20_LA_CHI,2025,2026-01-18,Soldier Field,CHI,LA,Ben Johnson,Sean McVay,17.0,17.0,...,0.0,1.0,1490.580105,1631.397360,62.146495,62.203513,0.059883,0.059941,0.4,0.6


In [54]:
df

,game_id,season,game_date,game_stadium,home_team,away_team,home_coach,away_coach,total_home_score,total_away_score,...,prev_home_avg_yards_gained,prev_away_avg_yards_gained,prev_home_team_glicko_rating,prev_away_team_glicko_rating,prev_home_team_rd,prev_away_team_rd,prev_home_team_vol,prev_away_team_vol,h2h_home_win_ratio,h2h_away_win_ratio
0,1999_01_ARI_PHI,1999,1999-09-12,Veterans Stadium,PHI,ARI,Andy Reid,Vince Tobin,24.0,25.0,...,0.0,1.0,1513.558175,1514.756649,68.080799,68.047382,0.059965,0.059965,0.0,0.0
1,1999_01_SF_JAX,1999,1999-09-12,Alltel Stadium,JAX,SF,Tom Coughlin,Steve Mariucci,41.0,3.0,...,0.0,1.0,1513.558175,1514.756649,68.080799,68.047382,0.059965,0.059965,0.0,0.0
2,1999_01_PIT_CLE,1999,1999-09-12,Cleveland Browns Stadium,CLE,PIT,Chris Palmer,Bill Cowher,0.0,43.0,...,0.0,1.0,1513.558175,1514.756649,68.080799,68.047382,0.059965,0.059965,0.0,0.0
3,1999_01_OAK_GB,1999,1999-09-12,Lambeau Field,GB,LV,Ray Rhodes,Jon Gruden,28.0,24.0,...,0.0,1.0,1513.558175,1514.756649,68.080799,68.047382,0.059965,0.059965,0.0,0.0
4,1999_01_NYG_TB,1999,1999-09-12,Raymond James Stadium,TB,NYG,Tony Dungy,Jim Fassel,13.0,17.0,...,0.0,1.0,1513.558175,1514.756649,68.080799,68.047382,0.059965,0.059965,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7458,2025_19_HOU_PIT,2025,2026-01-12,Acrisure Stadium,PIT,HOU,Mike Tomlin,DeMeco Ryans,6.0,30.0,...,0.0,0.0,1563.213624,1573.095618,61.339504,61.865037,0.059955,0.059929,0.6,0.4
7459,2025_20_BUF_DEN,2025,2026-01-17,Empower Field at Mile High,DEN,BUF,Sean Payton,Sean McDermott,30.0,30.0,...,0.0,1.0,1588.421870,1694.250626,61.855407,65.055276,0.059949,0.059935,0.2,0.8
7460,2025_20_SF_SEA,2025,2026-01-17,Lumen Field,SEA,SF,Mike Macdonald,Kyle Shanahan,41.0,6.0,...,1.0,1.0,1615.940716,1583.226667,61.780341,61.982002,0.059868,0.059895,0.4,0.6
7461,2025_20_LA_CHI,2025,2026-01-18,Soldier Field,CHI,LA,Ben Johnson,Sean McVay,17.0,17.0,...,0.0,1.0,1490.580105,1631.397360,62.146495,62.203513,0.059883,0.059941,0.4,0.6
